# Step 2: Launch AutoGluon Tabular Training

Uses `ModelTrainer` (SageMaker SDK v3) with the AWS-managed AutoGluon DLC image.

## Configuration

In [ ]:
import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-tabular"

AG_VERSION = "1.5"
PY_VERSION = "py312"

# Training configuration
INSTANCE_TYPE = "ml.m5.2xlarge"
JOB_NAME = "autogluon-tabular"

# S3 paths derived from BUCKET and S3_PREFIX
S3_TRAIN = f"s3://{BUCKET}/{S3_PREFIX}/processed/train/"
S3_TEST = f"s3://{BUCKET}/{S3_PREFIX}/processed/test/"
S3_CONFIG = f"s3://{BUCKET}/{S3_PREFIX}/config/"
S3_SERVING = ""  # Set to a serve.py S3 path to bundle inference code with model artifact
S3_OUTPUT = f"s3://{BUCKET}/{S3_PREFIX}/model/"

## Discover IAM role

In [ ]:
def get_role(role_arn=None):
    if role_arn:
        return role_arn
    iam = boto3.client("iam")
    paginator = iam.get_paginator("list_roles")
    for page in paginator.paginate():
        for role in page["Roles"]:
            name = role["RoleName"]
            if "SageMaker" in name or "sagemaker" in name:
                print(f"Discovered role: {role['Arn']}")
                return role["Arn"]
    raise ValueError("No SageMaker IAM role found. Pass role_arn explicitly.")

role_arn = get_role()

## Retrieve AutoGluon training image

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

session = Session()

image_uri = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="training",
    instance_type=INSTANCE_TYPE,
)
print(f"Image URI: {image_uri}")

## Build input data channels

In [ ]:
input_data_config = [
    {
        "channel_name": "train",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_TRAIN,
                "s3_data_type": "S3Prefix",
            }
        },
    },
    {
        "channel_name": "config",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_CONFIG,
                "s3_data_type": "S3Prefix",
            }
        },
    },
]

if S3_TEST:
    input_data_config.append({
        "channel_name": "test",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_TEST,
                "s3_data_type": "S3Prefix",
            }
        },
    })

if S3_SERVING:
    input_data_config.append({
        "channel_name": "serving",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_SERVING,
                "s3_data_type": "S3Prefix",
            }
        },
    })

print(f"Input channels: {[c['channel_name'] for c in input_data_config]}")

## Create ModelTrainer and launch training

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    SourceCode,
    OutputDataConfig,
    StoppingCondition,
)

trainer = ModelTrainer(
    training_image=image_uri,
    role=role_arn,
    source_code=SourceCode(
        source_dir=".",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        volume_size_in_gb=100,
        keep_alive_period_in_seconds=0,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=S3_OUTPUT,
    ),
    base_job_name=JOB_NAME,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    sagemaker_session=session,
)

training_job = trainer.train(
    input_data_config=input_data_config,
    wait=True,
    logs=True,
)

## Training job summary

In [ ]:
job_name = training_job.name
print(f"Training complete: {job_name}")
print(f"Console: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")